# SEN2SR — one small Sentinel-2 patch (save PNG)

Minimal demo: pick **one** original GeoTIFF under `**/S2/**`, take a **small center crop** (so CPU / notebook stays stable), run **ESA SEN2SRLite** (×4: 10 m → 2.5 m), write a **side-by-side PNG** to disk.

- **Does not** edit your source rasters (read-only).
- Uses the **Agg** matplotlib backend and **`savefig` + `plt.close()`** so the IDE/Jupyter does not have to render a huge canvas (often what stalls or crashes).
- For full tiles or many scenes, use a **CUDA** build of PyTorch on this machine (`torch.cuda.is_available()` should be True). CPU works on small crops only.

```bash
pip install sen2sr mlstac torch rasterio matplotlib numpy
```

CUDA example (adjust CUDA version to match your driver): install PyTorch from https://pytorch.org/get-started/locally/

Run all code cells top to bottom.

In [ ]:
# %pip install sen2sr mlstac torch rasterio matplotlib numpy

In [ ]:
print('hello world')

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.windows import Window
import torch

import mlstac
import sen2sr

In [ ]:
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

original_root = (
    PROJECT_ROOT.parent / "SDS_performance_analysis" / "data" / "sat_images"
)

MODEL_DIR = PROJECT_ROOT / "opensr_weights" / "SEN2SRLite_RGBN"
HF_MLM = "https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/NonReference_RGBN_x4/mlm.json"

OUTPUT_DIR = PROJECT_ROOT / "opensr_outputs"
OUTPUT_PNG = OUTPUT_DIR / "sen2sr_lr_vs_sr.png"

# Center crop size on the 10 m grid (pixels). Must be ≥128; multiples of 128 are tidy for SEN2SR tiling.
MAX_LR_SIDE = 256
PRED_OVERLAP = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def pick_one_s2_geotiff(root: Path) -> Path:
    cands = sorted(root.glob("**/S2/**/*.tif")) + sorted(root.glob("**/S2/**/*.tiff"))
    for p in cands:
        if p.is_file():
            return p
    raise FileNotFoundError(f"No S2 GeoTIFF under {root}")


scene_path = pick_one_s2_geotiff(original_root)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT.resolve())
print("original_root:", original_root.resolve())
print("scene_path   :", scene_path)
print("device       :", device)
print("torch CUDA?  :", torch.cuda.is_available())
print("LR crop cap  :", MAX_LR_SIDE, "px")
print("PNG out      :", OUTPUT_PNG.resolve())

PROJECT_ROOT : C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution
original_root: C:\Users\jnicolow\Documents\research\CRC\SDS_performance_analysis\data\sat_images
scene_path   : c:\Users\jnicolow\Documents\research\CRC\SDS_performance_analysis\data\sat_images\australianarrabeen\S2\S2_20151218_235302.tif
device       : cpu
torch CUDA?  : False
LR crop cap  : 256 px
PNG out      : C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\opensr_outputs\sen2sr_lr_vs_sr.png


In [3]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
mlm_path = MODEL_DIR / "mlm.json"

if not mlm_path.exists():
    print("Downloading SEN2SRLite (NonReference RGBN ×4) …")
    loader = mlstac.download(file=HF_MLM, output_dir=MODEL_DIR)
else:
    loader = mlstac.load(mlm_path.as_posix())

model = loader.compiled_model(device=device)
model.eval()
print("Model ready on", device)

Model ready on cpu


In [4]:
def _normalize_name(name: str) -> str:
    return "".join(ch for ch in name.lower() if ch.isalnum())


def _band_aliases() -> Dict[str, Tuple[str, ...]]:
    return {
        "blue": ("blue", "b", "band2", "b2"),
        "green": ("green", "g", "band3", "b3"),
        "red": ("red", "r", "band4", "b4"),
        "nir": ("nir", "nearinfrared", "nearir", "band5", "b5", "nir08", "nir1"),
    }


def extract_band_names(src: rasterio.io.DatasetReader) -> List[str]:
    names: List[str] = []
    for i in range(1, src.count + 1):
        desc = src.descriptions[i - 1] if src.descriptions else None
        tags = src.tags(i)
        tag_name = tags.get("name") or tags.get("band_name") or tags.get("long_name")
        chosen = (desc or tag_name or f"band{i}").strip()
        names.append(chosen)
    return names


def find_band_index(band_names: List[str], key: str) -> int:
    aliases = tuple(_normalize_name(a) for a in _band_aliases()[key])
    normalized = [_normalize_name(n) for n in band_names]
    for idx, n in enumerate(normalized):
        if n in aliases:
            return idx
    for idx, n in enumerate(normalized):
        if any(a in n for a in aliases):
            return idx
    raise ValueError(f"Could not find band '{key}' in names: {band_names}")


def to_reflectance_physical(data: np.ndarray) -> np.ndarray:
    mx = float(np.nanmax(data))
    if mx > 10:
        return (data / 10000.0).astype(np.float32)
    return np.clip(data, 0.0, 1.0).astype(np.float32)


def read_rgbn_center_crop(path: Path, max_side: int) -> Tuple[np.ndarray, int, int]:
    """Return (4, side, side) float reflectance stack and side length."""
    with rasterio.open(path) as src:
        if src.count < 4:
            raise ValueError(f"Need ≥4 bands, got {src.count}")
        h, w = src.height, src.width
        side = min(min(h, w), max_side)
        side = max(128, (side // 128) * 128)
        row_off = max(0, (h - side) // 2)
        col_off = max(0, (w - side) // 2)
        window = Window(col_off, row_off, side, side)
        data = src.read(window=window).astype(np.float32)
        names = extract_band_names(src)

    ri = find_band_index(names, "red")
    gi = find_band_index(names, "green")
    bi = find_band_index(names, "blue")
    ni = find_band_index(names, "nir")
    stacked = np.stack(
        [data[ri], data[gi], data[bi], data[ni]], axis=0,
    )
    stacked = to_reflectance_physical(stacked)
    return stacked, side, side


def rgb_composite(stack_4: np.ndarray) -> np.ndarray:
    r, g, b = stack_4[0], stack_4[1], stack_4[2]
    rgb = np.stack([r, g, b], axis=-1).astype(np.float64)
    out = np.zeros_like(rgb)
    for c in range(3):
        v = rgb[..., c]
        finite = np.isfinite(v)
        if not finite.any():
            continue
        lo, hi = np.percentile(v[finite], (2.0, 98.0))
        if hi <= lo:
            hi = lo + 1e-6
        out[..., c] = np.clip((v - lo) / (hi - lo), 0.0, 1.0)
    return out.astype(np.float32)


lr_stack, h_lr, w_lr = read_rgbn_center_crop(scene_path, MAX_LR_SIDE)
print("LR stack shape (C,H,W):", lr_stack.shape)

X = torch.from_numpy(lr_stack).float().to(device)
X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
with torch.inference_mode():
    sr_t = sen2sr.predict_large(X, model, overlap=PRED_OVERLAP)
sr = sr_t.detach().cpu().numpy()

print("SR shape (C,H,W):", sr.shape)
if sr.shape[1] != 4 * h_lr or sr.shape[2] != 4 * w_lr:
    warnings.warn(
        f"Unexpected SR shape {sr.shape} vs 4×LR {(h_lr, w_lr)} — check sen2sr / scene.",
        stacklevel=2,
    )

lr_rgb = rgb_composite(lr_stack)
sr_rgb = rgb_composite(sr)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(lr_rgb, interpolation="nearest")
axes[0].set_title(f"LR crop (~10 m)\n{h_lr}×{w_lr} px")
axes[0].axis("off")
axes[1].imshow(sr_rgb, interpolation="nearest")
axes[1].set_title("SEN2SR (~2.5 m)\n×4")
axes[1].axis("off")
fig.suptitle(scene_path.name, fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT_PNG, dpi=140, bbox_inches="tight")
plt.close(fig)

print("Saved:", OUTPUT_PNG.resolve())

LR stack shape (C,H,W): (4, 128, 128)


100%|██████████| 4/4 [00:00<00:00, 19.32it/s]

: 